# ResNet classifier

## Importations

In [1]:
import os
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.effects

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.utils import to_categorical

from classification.datasets import Dataset
from classification.utils.audio_student import AudioUtil, Feature_vector_DS
from classification.utils.plots import show_confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
np.random.seed(42)
tf.random.set_seed(42)

# visu
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

2026-03-14 21:28:43.732248: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-14 21:28:43.743650: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-14 21:28:44.141737: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-14 21:28:45.858904: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

TensorFlow version: 2.20.0
GPU Available: []


2026-03-14 21:28:46.421746: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Early stoping class

In [2]:
class GracefulStop(keras.callbacks.Callback):
    """
    Crée le fichier STOP_FILE pour arreter proprement apres l'epoch en cours.
    Le pipeline continue vers l'evaluation et la sauvegarde normalement.
    """
    STOP_FILE = "./stop_training.flag"

    def on_epoch_end(self, epoch, logs=None):
        if os.path.exists(self.STOP_FILE):
            print(f"\n  -> Fichier stop detecte, arret apres epoch {epoch+1}")
            os.remove(self.STOP_FILE)
            self.model.stop_training = True

## Config

In [ ]:
class Config:
    """
    Centralized configuration – PC training, no embedded size constraint.

    Paramètres audio
    ────────────────
    SAMPLE_RATE   : fréquence d'échantillonnage. 22050 Hz = standard qualité.
                    Capture les fréquences jusqu'à 11025 Hz (Nyquist).
                    Si tu baisses à 11025, tu coupes les hautes fréquences
                    mais tu divises par 2 la taille des spectrogrammes.

    N_MEL         : nombre de bandes de fréquence mel dans le spectrogramme.
                    20 = input 20xT (pauvre en information)
                    64 = input 64xT (3x plus riche, recommandé)
                    128 = encore plus riche, mais plus lent

    N_FFT         : taille de la fenêtre FFT.
                    512  = 23ms par frame à 22050Hz, résolution temporelle fine
                    1024 = 46ms par frame, meilleure résolution fréquentielle
                    2048 = 93ms par frame, très bonne résolution fréq.
                    Compromis : 1024 est le standard pour sons environnementaux

    HOP_LENGTH    : pas entre deux frames FFT successives.
                    Plus petit = plus de frames (plus de données, plus lent)
                    Plus grand = moins de frames
                    Règle : N_FFT // 4 est un bon défaut

    DURATION_MS   : durée de la fenêtre d'analyse en ms.
                    Doit être > durée du son le plus court de tes classes.
                    Gunshot : ~200ms. Fire : continu. 1000ms est un bon compromis.

    Paramètres d'entraînement
    ─────────────────────────
    BATCH_SIZE    : exemples traités ensemble avant une mise à jour des poids.
                    32–64 : bruité mais bonne généralisation
                    128–256 : stable, converge vite
                    Avec petit dataset → 32 ou 64

    LEARNING_RATE : pas d'ajustement des poids. Avec Adam :
                    1e-3 = standard, peut sauter par-dessus les optima
                    5e-4 = plus conservateur, souvent meilleur avec petit LR decay
                    1e-4 = très lent mais précis
                    On utilise cosine decay donc LR descend automatiquement.

    DROPOUT_RATE  : fraction des neurones éteints pendant l'entraînement.
                    0.3 = peu de régularisation
                    0.5 = fort, bien pour petit dataset
                    0.6+ = risque d'underfitting

    LABEL_SMOOTHING : "ramollit" les labels one-hot.
                    0.0 = [0, 0, 1, 0] (dur, le modèle est trop sûr de lui)
                    0.1 = [0.025, 0.025, 0.925, 0.025] (plus humble)
                    Valeurs typiques : 0.05–0.15

    MIXUP_ALPHA   : force du mixup. 0 = désactivé.
                    0.2–0.4 = léger mélange, recommandé pour petit dataset
                    Plus grand → exemples plus "hybrides", risque de confusion

    TTA_STEPS     : nombre de versions augmentées à l'inférence (Test-Time Aug).
                    1 = pas de TTA
                    5 = bon équilibre qualité/vitesse
                    10 = +marginal, 2 fois plus lent
    """

    # ── Audio ─────────────────────────────────────────────────
    SAMPLE_RATE  = 11025   # Hz
    N_MEL        = 64      # bandes mel  (était 20, 3 fois plus riche)
    N_FFT        = 512    # taille FFT  (était 512)
    HOP_LENGTH   = 128     # pas FFT     (N_FFT // 4)
    DURATION_MS  = 1000    # ms

    # ── Augmentation flags ────────────────────────────────────
    AUG_TIME_SHIFT   = True
    AUG_PITCH_SHIFT  = True
    AUG_TIME_STRETCH = True
    AUG_NOISE        = True
    AUG_SPEC_MASKING = True

    # ── Augmentation params ───────────────────────────────────
    PITCH_SHIFT_RANGE  = (-3, 3)    # +/- 3 demi-tons (plus large qu'avant)
    TIME_STRETCH_RANGE = (0.85, 1.15)
    NOISE_SIGMA        = 0.04

    # ── Architecture ──────────────────────────────────────────
    DROPOUT_RATE    = 0.5
    L2_REG          = 1e-4   # régularisation L2 sur les poids des Conv
    LABEL_SMOOTHING = 0.1    # lissage des labels
    MIXUP_ALPHA     = 0.3    # force du mixup (0 = désactivé)

    # ── Training ──────────────────────────────────────────────
    BATCH_SIZE    = 32
    EPOCHS        = 300
    LEARNING_RATE = 4e-4

    # ── Callbacks ─────────────────────────────────────────────
    EARLY_STOPPING_PATIENCE = 60
    WARMUP_EPOCHS           = 10   # epochs sans LR decay au départ

    # ── TTA ───────────────────────────────────────────────────
    TTA_STEPS = 5   # nombre de passes augmentées à l'inférence

    # ── Paths ─────────────────────────────────────────────────
    MODEL_DIR = "./data/models/models_resnet/"

    # ── Données réelles ───────────────────────────────────────
    TEST_ON_ACQUIRED_DATA = True
    USE_EXTRA_TRAIN_DATA  = True
    ACQUIRED_DATA         = "../mcu/hands_on_audio_acquisition/audio_files"
    REAL_SUBFOLDER        = ["training"]
    EXTRA_TRAIN_DIR       = "../mcu/hands_on_audio_acquisition/audio_files"
    EXTRA_AUG_PASSES      = 60
    TRAIN_VAL_SPLIT   = 0.8    
    AUG_VAL           = True 
    RANDOM_SEED       = 42
    # ── Decision ──────────────────────────────────────────────
    CONFIDENCE_THRESHOLD  = 0.6

config = Config()


## Augmentation

In [ ]:
class AdvancedAudioAugmentation:
    @staticmethod
    def time_shift(audio, shift_max=0.2):
        sig, sr = audio
        shift = int(np.random.uniform(-shift_max, shift_max) * len(sig))
        sig_shifted = np.zeros_like(sig)
        if shift > 0:
            sig_shifted[shift:] = sig[:-shift]
        elif shift < 0:
            sig_shifted[:shift] = sig[-shift:]
        else:
            sig_shifted = sig.copy()
        return (sig_shifted, sr)
    
    @staticmethod
    def pitch_shift(audio, n_steps_range=(-1.5, 1.5)):
        sig, sr = audio
        n = np.random.uniform(*n_steps_range)
        return (librosa.effects.pitch_shift(sig, sr=sr, n_steps=n), sr)

    @staticmethod
    def time_stretch(audio, rate_range=(0.85, 1.15)):
        sig, sr = audio
        rate = np.random.uniform(*rate_range)
        stretched = librosa.effects.time_stretch(sig, rate=rate)
        if len(stretched) > len(sig):   stretched = stretched[:len(sig)]
        elif len(stretched) < len(sig): stretched = np.pad(stretched, (0, len(sig)-len(stretched)))
        return (stretched, sr)

    @staticmethod
    def apply_pipeline(audio, config, skip_noise=False):
        """Pipeline complet. skip_noise=True pour données réelles."""
        if config.AUG_TIME_SHIFT   and np.random.random() > 0.5:
            audio = AdvancedAudioAugmentation.time_shift(audio)
        if config.AUG_PITCH_SHIFT  and np.random.random() > 0.5:
            audio = AdvancedAudioAugmentation.pitch_shift(audio, config.PITCH_SHIFT_RANGE)
        if config.AUG_TIME_STRETCH and np.random.random() > 0.5:
            audio = AdvancedAudioAugmentation.time_stretch(audio, config.TIME_STRETCH_RANGE)
            
        if not skip_noise and config.AUG_NOISE and np.random.random() > 0.5:
            audio = AudioUtil.add_noise(audio, sigma=config.NOISE_SIGMA)
        return audio

class AugmentedFeatureVectorDS(Feature_vector_DS):
    """Extended Dataset with advanced augmentation + FIXES"""

    def __init__(self, *args, use_advanced_aug=False, config=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_advanced_aug = use_advanced_aug
        self.config = config or Config()

    def get_audiosignal(self, cls_index):
        audio_file = self.dataset[cls_index]
        aud = AudioUtil.open(audio_file)
        aud = AudioUtil.resample(aud, self.sr)
        if self.use_advanced_aug:
            aud = AdvancedAudioAugmentation.apply_pipeline(
                aud, self.config)
        if self.data_aug is not None:
            if "add_bg" in self.data_aug:
                aud = AudioUtil.add_bg(aud, self.dataset, num_sources=1,
                                       max_ms=self.duration, amplitude_limit=0.1)
            if "echo"    in self.data_aug: aud = AudioUtil.echo(aud)
            if "noise"   in self.data_aug: aud = AudioUtil.add_noise(aud, sigma=0.05)
            if "scaling" in self.data_aug: aud = AudioUtil.scaling(aud, scaling_limit=5)
        aud = (aud[0] / (np.max(np.abs(aud[0])) + 1e-8), aud[1])
        return aud

    def __getitem__(self, cls_index):
        aud   = self.get_audiosignal(cls_index)
        sgram = AudioUtil.melspectrogram(aud, Nmel=self.nmel, Nft=self.Nft)
        if self.use_advanced_aug and self.config.AUG_SPEC_MASKING:
            if np.random.random() > 0.5:
                sgram = AudioUtil.spectro_aug_timefreq_masking(
                    sgram, max_mask_pct=0.1, n_freq_masks=2, n_time_masks=2)
        return sgram

    def treat_spec(self, sgram):
        n_cols = sgram.shape[1]
        if n_cols < self.ncol:
            pad_width = self.ncol - n_cols
            sgram = np.pad(sgram, ((0, 0), (0, pad_width)),
                           mode='constant', constant_values=0)
            n_cols = sgram.shape[1]
        indexes = np.arange(0, n_cols - self.ncol + 1, self.step, dtype=int)
        if len(indexes) == 0:
            indexes = np.array([0])
        sgrams = []
        for i in indexes:
            window = sgram[:, i: i + self.ncol]
            if window.shape[1] < self.ncol:
                pad_width = self.ncol - window.shape[1]
                window = np.pad(window, ((0, 0), (0, pad_width)),
                                mode='constant', constant_values=0)
            sgrams.append(window)
        sgrams = np.array(sgrams)
        fv = sgrams.reshape(sgrams.shape[0], -1)
        if self.normalize:
            norms = np.linalg.norm(fv, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            fv = fv / norms
        if self.pca is not None:
            fv = np.array([self.pca.transform([i])[0] for i in fv])
        return fv

## Feature extraction

In [ ]:
class FeatureExtractor:
    """
    Extrait un mel-spectrogramme (N_MEL x T) depuis un fichier audio,
    puis le découpe en fenêtres via sliding window.


    """

    def __init__(self, config):
        self.config = config
        # Nombre de frames dans une fenêtre de DURATION_MS
        self.n_frames = int(
            config.DURATION_MS / 1000 * config.SAMPLE_RATE / config.HOP_LENGTH
        ) + 1
        # Step = 50% chevauchement par défaut
        self.step_frames = max(1, self.n_frames // 2)

    def extract_from_audio(self, audio_tuple, augment=False, skip_noise=False):
        """
        audio_tuple : (signal_numpy, sample_rate)
        Retourne    : liste de feature vectors aplatis (1D)
        """
        sig, sr = audio_tuple

        # Augmentation (optionnelle)
        if augment:
            sig, sr = AudioAugmentation.apply_pipeline(
                (sig, sr), self.config, skip_noise=skip_noise)

        # Normalisation amplitude
        sig = sig / (np.max(np.abs(sig)) + 1e-8)

        # Mel-spectrogramme log
        mel = librosa.feature.melspectrogram(
            y=sig, sr=sr,
            n_mels=self.config.N_MEL,
            n_fft=self.config.N_FFT,
            hop_length=self.config.HOP_LENGTH
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)  # shape: (N_MEL, T)

        # Normalisation spectrale (mean/std par fréquence)
        log_mel = (log_mel - log_mel.mean(axis=1, keepdims=True)) / \
                  (log_mel.std(axis=1, keepdims=True) + 1e-8)

        # Padding si trop court
        T = log_mel.shape[1]
        if T < self.n_frames:
            log_mel = np.pad(log_mel, ((0,0),(0, self.n_frames - T)))
            T = self.n_frames

        # Spec masking (après normalisation)
        if augment and self.config.AUG_SPEC_MASKING and np.random.random() > 0.5:
            log_mel = AudioUtil.spectro_aug_timefreq_masking(
                log_mel, max_mask_pct=0.15, n_freq_masks=3, n_time_masks=3)

        # Sliding window → liste de fenêtres
        windows = []
        for start in range(0, T - self.n_frames + 1, self.step_frames):
            window = log_mel[:, start:start + self.n_frames]   # (N_MEL, n_frames)
            windows.append(window)

        if len(windows) == 0:
            windows.append(log_mel[:, :self.n_frames])

        return windows   # liste de (N_MEL, n_frames)

    def extract_from_file(self, path, augment=False, skip_noise=False):
        aud = AudioUtil.open(path)
        aud = AudioUtil.resample(aud, self.config.SAMPLE_RATE)
        return self.extract_from_audio(aud, augment=augment, skip_noise=skip_noise)

    @property
    def input_shape(self):
        """Shape d'un exemple : (N_MEL, n_frames, 1) — format image 2D pour la CNN"""
        return (self.config.N_MEL, self.n_frames, 1)

## Data preparation

In [ ]:
def load_from_dir(folder, classnames, label_to_idx, extractor,
                  n_aug_passes=0, subfolder=None, skip_noise=False):
    X_list, y_list = [], []
    for classname in classnames:
        path = os.path.join(folder, classname, subfolder) if subfolder \
               else os.path.join(folder, classname)
        if not os.path.exists(path):
            print(f"    ! {path} introuvable - ignore")
            continue
        files = sorted([f for f in os.listdir(path) if f.endswith('.wav')])
        print(f"    {classname:<15}: {len(files):>4} fichiers")
        for f in files:
            fp = os.path.join(path, f)
            for w in extractor.extract_from_file(fp, augment=False):
                X_list.append(w)
                y_list.append(label_to_idx[classname])
            for _ in range(n_aug_passes):
                for w in extractor.extract_from_file(fp, augment=True, skip_noise=skip_noise):
                    X_list.append(w)
                    y_list.append(label_to_idx[classname])
    X = np.array(X_list)[..., np.newaxis]
    y = np.array(y_list)
    return X, y


def _synth_to_resnet_format(X_flat, y_str, classnames, extractor):
    label_to_idx = {c: i for i, c in enumerate(classnames)}
    n_mel    = extractor.config.N_MEL
    n_frames = extractor.n_frames
    expected = n_mel * n_frames
    X_list, y_list = [], []
    for vec, label in zip(X_flat, y_str):
        if len(vec) == expected:
            window = vec.reshape(n_mel, n_frames)
        else:
            window_flat = np.interp(
                np.linspace(0, len(vec)-1, expected),
                np.arange(len(vec)), vec
            )
            window = window_flat.reshape(n_mel, n_frames)
        X_list.append(window)
        y_list.append(label_to_idx[label])
    X = np.array(X_list)[..., np.newaxis]
    y = np.array(y_list)
    return X, y


def prepare_dataset(dataset, config, extractor, n_augmentations=60):
    classnames   = dataset.list_classes()
    label_to_idx = {c: i for i, c in enumerate(classnames)}

    print("\n" + "="*70)
    print("PREPARA DES DONNEES (split)")
    print("="*70)

    # 1. Charger tous les fichiers reels
    X_real_parts, y_real_parts = [], []
    if config.USE_EXTRA_TRAIN_DATA:
        subfolders = getattr(config, 'REAL_SUBFOLDERS', ["training"])
        for subfolder in subfolders:
            path_exists = any(
                os.path.exists(os.path.join(config.EXTRA_TRAIN_DIR, cls, subfolder))
                for cls in classnames
            )
            if not path_exists:
                print(f"  ! Dossier {subfolder}/ introuvable - ignore")
                continue
            print(f"\n[Reel {subfolder}/]")
            X_sub, y_sub = load_from_dir(
                config.EXTRA_TRAIN_DIR, classnames, label_to_idx, extractor,
                n_aug_passes=0, subfolder=subfolder, skip_noise=True
            )
            X_real_parts.append(X_sub)
            y_real_parts.append(y_sub)

    # 2. Charger les fichiers synthetiques BRUTS
    print(f"\n[Synthetique - original sans augmentation]")
    ds_orig = AugmentedFeatureVectorDS(
        dataset, Nft=config.N_FFT, nmel=config.N_MEL,
        duration=config.DURATION_MS, step=config.DURATION_MS // 2,
        use_advanced_aug=False, config=config
    )
    X_flat, y_str = ds_orig.get_feature_vectors()
    X_synth, y_synth = _synth_to_resnet_format(X_flat, y_str, classnames, extractor)
    print(f"  Extrait : {len(X_synth)} vecteurs  shape={X_synth.shape[1:]}")

    # ── 2. Test set ───────────────────────────────────────────
    if config.TEST_ON_ACQUIRED_DATA:
        print(f"\n[TEST - réel] {config.ACQUIRED_DATA}/classname/test/")
        X_test, y_test = load_from_dir(
            config.ACQUIRED_DATA, classnames, label_to_idx, extractor,
            n_aug_passes=0, subfolder="test"
        )
        X_test = X_test.astype("float32")#########################################################################
        y_test = y_test.astype("int32")############################################################################
        X_train_base, y_train_base = X_synth.copy(), y_synth.copy()
    else:
        X_train_base, X_test, y_train_base, y_test = train_test_split(
            X_synth, y_synth, test_size=0.2, stratify=y_synth, random_state=42)
        print(f"  Split 80/20 → train={len(X_train_base)} test={len(X_test)}")

    # ── 3. Validation set ─────────────────────────────────────
    if config.USE_REAL_VAL and config.TEST_ON_ACQUIRED_DATA:
        print(f"\n[VAL - réel] {config.ACQUIRED_DATA}/classname/val/")
        X_val, y_val = load_from_dir(
            config.ACQUIRED_DATA, classnames, label_to_idx, extractor,
            n_aug_passes=0, subfolder="val"
        )
    else:
        X_train_base, X_val, y_train_base, y_val = train_test_split(
            X_train_base, y_train_base, test_size=0.2,
            stratify=y_train_base, random_state=42)
        print(f"  Val synthétique : {len(X_val)} vecteurs")

    # ── 4. Extra training data réel ───────────────────────────
    if config.USE_EXTRA_TRAIN_DATA:
        extra_passes = getattr(config, 'EXTRA_AUG_PASSES', n_augmentations)
        print(f"\n[TRAIN+ - réel] {config.EXTRA_TRAIN_DIR}/classname/train/")
        print(f"  Augmentation : {extra_passes} passes (sans noise)")
        X_extra, y_extra = load_from_dir(
            config.EXTRA_TRAIN_DIR, classnames, label_to_idx, extractor,
            n_aug_passes=extra_passes, subfolder="train", skip_noise=True
        )
        print(f"  → {len(X_extra)} vecteurs après augmentation")
        X_train_base = np.concatenate([X_train_base, X_extra])
        y_train_base = np.concatenate([y_train_base, y_extra])

    # 4. Split 80/20 SUR LES ORIGINAUX
    #    Chaque fichier original va soit en train soit en val, jamais les deux
    #    => pas de data leakage entre versions augmentees
    X_orig_train, X_orig_val, y_orig_train, y_orig_val = train_test_split(
        X_raw, y_raw,
        test_size    = 1 - config.TRAIN_VAL_SPLIT,
        stratify     = y_raw,
        random_state = config.RANDOM_SEED
    )
    print(f"  -> Originaux train : {len(X_orig_train)}  |  val : {len(X_orig_val)}")

    # 5. Augmenter la partie train
    print(f"\n[Augmentation train - {n_augmentations} passes]")
    X_aug_list, y_aug_list = [X_orig_train], [y_orig_train]

    for pass_i in range(n_augmentations):
        print(f"  Pass {pass_i+1}/{n_augmentations}...", end="\r")
        ds_aug = AugmentedFeatureVectorDS(
            dataset, Nft=config.N_FFT, nmel=config.N_MEL,
            duration=config.DURATION_MS, step=config.DURATION_MS // 2,
            use_advanced_aug=True, config=config
        )
        X_flat_aug, y_str_aug = ds_aug.get_feature_vectors()
        Xp, yp = _synth_to_resnet_format(X_flat_aug, y_str_aug, classnames, extractor)

        if config.TEST_ON_ACQUIRED_DATA:
            X_aug_list.append(Xp)
            y_aug_list.append(yp)
        else:
            Xp_tr, _, yp_tr, _ = train_test_split(
                Xp, yp, test_size=0.2, stratify=yp, random_state=42)
            X_aug_list.append(Xp_tr)
            y_aug_list.append(yp_tr)

    X_train = np.concatenate(X_aug_list)
    y_train  = np.concatenate(y_aug_list)
    print()

    # 6. Val = val SANS augmentation
    X_val, y_val = X_orig_val, y_orig_val
    print(f"  -> Val (originaux purs)     : {len(X_val)}")

    # 7. Test set - jamais dans le pool
    print(f"\n[Test reel]")
    X_test, y_test = load_from_dir(
        config.ACQUIRED_DATA, classnames, label_to_idx, extractor,
        n_aug_passes=0, subfolder="test"
    )

    # Resume
    print("\n" + "="*70)
    print("RESUME")
    print("="*70)
    for name, y_arr in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
        counts = Counter(y_arr)
        print(f"\n{name} ({len(y_arr)} vecteurs):")
        for i, c in enumerate(classnames):
            print(f"  {c:<15}: {counts.get(i, 0):>6}")

    return X_train, X_val, X_test, y_train, y_val, y_test, classnames

## Architecture | ResNet-small audio

In [ ]:
def residual_block(x, filters, stride=1, l2=1e-4):
    """
    Bloc résiduel :
        y = F(x) + x
    Si les dimensions changent (stride ou filters différents),
    on projette x avec une Conv 1x1 pour que l'addition soit possible.

    Pourquoi ça aide ?
    ──────────────────
    Sans résidu : gradient doit traverser toutes les couches → s'atténue
    Avec résidu : gradient peut "court-circuiter" → apprentissage plus stable
    """
    shortcut = x
    reg = regularizers.l2(l2)

    # Branche principale
    x = layers.Conv2D(filters, 3, strides=stride, padding='same',
                      kernel_regularizer=reg, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, padding='same',
                      kernel_regularizer=reg, use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    # Projection du shortcut si nécessaire
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 kernel_regularizer=reg, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def build_resnet_audio(input_shape, num_classes, config):
    """
    Architecture ResNet-small adaptée à la classification audio.

    input_shape : (N_MEL, n_frames, 1)  ex: (64, 87, 1)

    Flux des données :
    ──────────────────
    (64, 87, 1)
    → StemConv(32)      : (64, 87, 32)   capture les patterns locaux basiques
    → MaxPool           : (32, 43, 32)   réduit la taille, garde l'essentiel
    → ResBlock(64) x2   : (32, 43, 64)   features intermédiaires
    → ResBlock(128, s=2): (16, 22, 128)  features complexes, résolution réduite
    → ResBlock(128) x1  : (16, 22, 128)
    → ResBlock(256, s=2): (8,  11, 256)  features abstraites
    → GlobalAvgPool     : (256,)         résume tout le spectrogramme en 256 valeurs
    → Dense(256) + Drop : (256,)
    → Softmax(4)        : (4,)
    """
    inp = layers.Input(shape=input_shape)
    reg = regularizers.l2(config.L2_REG)

    # Stem, première convolution large
    x = layers.Conv2D(32, 5, padding='same', kernel_regularizer=reg, use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPool2D(2, padding='same')(x)   # (N_MEL//2, T//2, 32)

    # Blocs résiduels
    x = residual_block(x, 64,  stride=1, l2=config.L2_REG)
    x = residual_block(x, 64,  stride=1, l2=config.L2_REG)
    x = residual_block(x, 128, stride=2, l2=config.L2_REG)   
    x = residual_block(x, 128, stride=1, l2=config.L2_REG)
    x = residual_block(x, 256, stride=2, l2=config.L2_REG)   

    # Tête de classification
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(config.DROPOUT_RATE)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out, name="ResNet_Audio")

    # Label smoothing intégré dans la loss
    # Au lieu de [0,0,1,0] le modèle reçoit [0.033, 0.033, 0.9, 0.033]
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
        loss=keras.losses.CategoricalCrossentropy(
            label_smoothing=config.LABEL_SMOOTHING),
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2)]
    )

    model.summary()
    params = model.count_params()
    print(f"\nParamètres : {params:,}  (~{params*4/1024:.0f} KB FP32)")
    return model


## MixUp
"C'est contre-intuitif mais extrêmement efficace sur petit dataset." Créer des exemples impossibles (30% chainsaw + 70% fire) et le modèle apprend que la frontière entre les classes est floue. Résultat : il généralise mieux sur des sons "intermédiaires" comme ceux captés sur le micro embarqués

In [9]:
def mixup_batch(X, y, alpha=0.3):
    """
    MixUp : crée des exemples synthétiques en interpolant deux exemples réels.

    Exemple avec alpha=0.3 :
        lambda ~ Beta(0.3, 0.3) → valeur entre 0 et 1
        X_mix = 0.7 * X[i] + 0.3 * X[j]
        y_mix = 0.7 * y[i] + 0.3 * y[j]   (labels "mous")

    Effet : le modèle apprend des frontières de décision plus douces
    → moins d'overfit, meilleure calibration des probabilités
    """
    if alpha == 0:
        return X, y
    lam = np.random.beta(alpha, alpha)
    idx = np.random.permutation(len(X))
    X_mix = lam * X + (1 - lam) * X[idx]
    y_mix = lam * y + (1 - lam) * y[idx]
    return X_mix, y_mix


class MixupDataGenerator(keras.utils.Sequence):
    """
    Générateur Keras qui applique MixUp à chaque batch.
    Keras appelle __getitem__(batch_idx) à chaque step.
    """
    def __init__(self, X, y, batch_size, num_classes, alpha=0.3, shuffle=True):
        self.X, self.y_int = X, y
        self.y_cat  = to_categorical(y, num_classes)
        self.bs     = batch_size
        self.alpha  = alpha
        self.shuffle = shuffle
        self.idx    = np.arange(len(X))

    def __len__(self):
        return int(np.ceil(len(self.X) / self.bs))

    def __getitem__(self, i):
        batch_idx = self.idx[i*self.bs:(i+1)*self.bs]
        Xb = self.X[batch_idx]
        yb = self.y_cat[batch_idx]
        Xb, yb = mixup_batch(Xb, yb, self.alpha)
        return Xb, yb

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)

## Training

In [9]:
def train_model(model, X_train, y_train, X_val, y_val, config, num_classes):

    train_gen = MixupDataGenerator(
        X_train, y_train, config.BATCH_SIZE, num_classes,
        alpha=config.MIXUP_ALPHA
    )
    y_val_cat = to_categorical(y_val, num_classes)
    steps_per_epoch = len(train_gen)
    #CosineDecayRestarts peut être une bonne piste mais il faut mieux calibrer le min, max et la longueur des cycles (regarder le run resnet_better_lr0.0005_bs32_03_13_20:55 pour calibrer au niveau des plateaux)
    RESTART_EPOCHS = 45 # 40 était un peu trop rapide
    lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate = config.LEARNING_RATE, # je mettrais 4e-4
        first_decay_steps     = steps_per_epoch * RESTART_EPOCHS,
        t_mul = 1.0,    
        m_mul = 0.7,   
        alpha = 1e-6    
    )
    model.optimizer.learning_rate = lr_schedule

    # Callbacks — monitor val_loss plutôt que val_accuracy (moins bruité)
    cbs = [
        callbacks.EarlyStopping(
            monitor              = 'val_loss',   
            patience             = config.EARLY_STOPPING_PATIENCE,
            restore_best_weights = True,
            verbose              = 1
        ),
        callbacks.ModelCheckpoint(
            os.path.join(config.MODEL_DIR, 'best_model.keras'),
            monitor      = 'val_loss',           
            save_best_only = True,
            verbose      = 1
        ),
        callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,       
        patience = 10,    
        min_lr   = 1e-6,
        verbose  = 1
        ),
        WandbMetricsLogger(log_freq='epoch'),
        GracefulStop(),
    ]
    print("\n" + "="*70)
    print("ENTRAÎNEMENT")
    print("="*70)
    print(f"  Train : {len(X_train)} exemples | {len(train_gen)} batches/epoch")
    print(f"  Val   : {len(X_val)} exemples")
    print(f"  MixUp alpha={config.MIXUP_ALPHA} | Label smoothing={config.LABEL_SMOOTHING}")

    history = model.fit(
        train_gen,
        validation_data=(X_val, y_val_cat),
        epochs=config.EPOCHS,
        callbacks=cbs,
        verbose=1
    )

    with open(os.path.join(config.MODEL_DIR, 'history.pkl'), 'wb') as f:
        pickle.dump(history.history, f)

    return history

## TTA (test time augmentation)

In [11]:
def predict_with_tta(model, X, extractor, config, n_steps=5):
    """
    Test-Time Augmentation :
    ─────────────────────────
    Au lieu de prédire une seule fois sur X,
    on prédit n_steps fois sur X légèrement modifié (augmentation légère)
    puis on moyenne les probabilités.

    Pourquoi ça marche ?
    ──────────────────────
    Chaque version augmentée met en évidence un aspect différent du son.
    La moyenne réduit la variance de la prédiction = +2-3% accuracy typique.
    """
    # Prédiction sur l'original
    all_probs = model.predict(X, verbose=0)

    # Prédictions sur versions augmentées
    for _ in range(n_steps - 1):
        # Légère augmentation (time shift + spec masking seulement)
        X_aug = X.copy()
        for i in range(len(X_aug)):
            # Time shift aléatoire sur l'axe temporal (axis=1)
            shift = np.random.randint(-3, 4)
            X_aug[i, :, :, 0] = np.roll(X_aug[i, :, :, 0], shift, axis=1)
        all_probs += model.predict(X_aug, verbose=0)

    return all_probs / n_steps   # moyenne des probabilités

In [12]:
def plot_history(history, config):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
        ax.plot(history.history[metric],          label='Train', lw=2)
        ax.plot(history.history[f'val_{metric}'], label='Val',   lw=2)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(alpha=0.3)

    best = np.argmax(history.history['val_accuracy'])
    axes[0].axvline(best, color='red', linestyle='--', alpha=0.5, label=f'best epoch {best+1}')
    axes[0].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(config.MODEL_DIR, 'training_history.png'), dpi=150)
    plt.show()
    print(f"\nMeilleure val_accuracy : {100*history.history['val_accuracy'][best]:.2f}% (epoch {best+1})")


def plot_per_class_metrics(classnames, prec, rec, f1, sup):
    x = np.arange(len(classnames))
    width = 0.25
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Barres precision / recall / F1
    ax = axes[0]
    ax.bar(x - width, prec, width, label='Precision', color='#2196F3')
    ax.bar(x,         rec,  width, label='Recall',    color='#4CAF50')
    ax.bar(x + width, f1,   width, label='F1',        color='#9C27B0')
    ax.set_xticks(x)
    ax.set_xticklabels(classnames, rotation=20, ha='right')
    ax.set_ylim(0, 1.1)
    ax.set_title('Precision / Recall / F1 par classe', fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for i, (p, r, f) in enumerate(zip(prec, rec, f1)):
        ax.text(i - width, p + 0.02, f'{p:.2f}', ha='center', fontsize=9)
        ax.text(i,         r + 0.02, f'{r:.2f}', ha='center', fontsize=9)
        ax.text(i + width, f + 0.02, f'{f:.2f}', ha='center', fontsize=9)

    # Support (nombre d'exemples par classe)
    ax2 = axes[1]
    bars = ax2.bar(classnames, sup, color='#FF9800', alpha=0.8)
    ax2.set_title('Support par classe (test set)', fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    for bar, s in zip(bars, sup):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(int(s)), ha='center', fontsize=11)

    plt.tight_layout()
    plt.show()

## Evaluation

In [13]:
def evaluate_full(model, X_train, y_train, X_val, y_val, X_test, y_test,
                  classnames, config, extractor):
    n_classes = len(classnames)
    y_train_cat = to_categorical(y_train, n_classes)
    y_val_cat   = to_categorical(y_val,   n_classes)
    y_test_cat  = to_categorical(y_test,  n_classes)

    print("\n" + "="*70)
    print("ÉVALUATION COMPLÈTE")
    print("="*70)

    # Prédictions (TTA sur test uniquement)
    y_train_probs = model.predict(X_train, verbose=0)
    y_val_probs   = model.predict(X_val,   verbose=0)
    y_test_probs  = predict_with_tta(model, X_test, extractor, config,
                                     n_steps=config.TTA_STEPS)

    y_train_pred = np.argmax(y_train_probs, axis=1)
    y_val_pred   = np.argmax(y_val_probs,   axis=1)
    y_test_pred  = np.argmax(y_test_probs,  axis=1)

    acc_train = np.mean(y_train_pred == y_train)
    acc_val   = np.mean(y_val_pred   == y_val)
    acc_test  = np.mean(y_test_pred  == y_test)

    print(f"\n  Train : {100*acc_train:.2f}%")
    print(f"  Val   : {100*acc_val:.2f}%   (mixed pool 20%)") 
    print(f"  Test  : {100*acc_test:.2f}%   (REAL + TTAx{config.TTA_STEPS})")
    print(f"\n  Gap Train→Test : {100*(acc_train-acc_test):.2f}%")

    if   acc_train - acc_val > 0.15: print("  ! Overfitting significatif (>15%)")
    elif acc_train - acc_val > 0.08: print("  ! Overfitting modéré (>8%)")
    else:                            print("  :)) Bon équilibre train/val")

    # 3 matrices de confusion
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    sets = [
        ("Training",   y_train, y_train_pred, acc_train),
        ("Validation", y_val,   y_val_pred,   acc_val),
        ("Test (TTA)", y_test,  y_test_pred,  acc_test),
    ]
    for ax, (title, y_true, y_pred, acc) in zip(axes, sets):
        cm = confusion_matrix(y_true, y_pred)
        im = ax.imshow(cm, cmap='Blues')
        ax.set_title(f'{title}\n{100*acc:.1f}%', fontsize=14, fontweight='bold')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ticks = np.arange(n_classes)
        ax.set_xticks(ticks); ax.set_xticklabels(classnames, rotation=45, ha='right')
        ax.set_yticks(ticks); ax.set_yticklabels(classnames)
        thresh = cm.max() / 2
        for i in range(n_classes):
            for j in range(n_classes):
                ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                        color='white' if cm[i,j] > thresh else 'black')
        ax.set_xlabel('Prédit'); ax.set_ylabel('Vrai')

    plt.suptitle("Matrices de confusion – Train / Val / Test",
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(config.MODEL_DIR, 'confusion_matrices.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

    # ── W&B : 3 confusion matrices interactives ───────────────────────────
    # wandb.plot.confusion_matrix génère un plot cliquable dans l'UI W&B
    # avec drill-down par classe — bien plus utile que l'image statique
    for split_name, y_true, y_pred in [
        ("train", y_train, y_train_pred),
        ("val",   y_val,   y_val_pred),
        ("test",  y_test,  y_test_pred),
    ]:
        wandb.log({
            f"confusion_matrix/{split_name}": wandb.plot.confusion_matrix(
                y_true      = y_true.tolist(),
                preds       = y_pred.tolist(),
                class_names = classnames,
                title       = f"Confusion — {split_name}",
            )
        })
    # Métriques par classe
    print(f"\n{'Classe':<15} {'Précision':>10} {'Rappel':>10} {'F1':>8} {'Support':>10}")
    print("-" * 60)
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_test, y_test_pred, labels=range(n_classes), average=None)
    for i, c in enumerate(classnames):
        print(f"{c:<15} {prec[i]:>10.3f} {rec[i]:>10.3f} {f1[i]:>8.3f} {int(sup[i]):>10}")
    print("-" * 60)
    print(f"{'Macro Avg':<15} {prec.mean():>10.3f} {rec.mean():>10.3f} "
          f"{f1.mean():>8.3f} {int(sup.sum()):>10}")
    class_table = wandb.Table(columns=["classe", "precision", "recall", "f1", "support"])
    for i, c in enumerate(classnames):
        class_table.add_data(c, round(float(prec[i]),3), round(float(rec[i]),3),
                             round(float(f1[i]),3), int(sup[i]))
    wandb.log({"metrics_par_classe": class_table})

    
    return {
        'train': acc_train, 'val': acc_val, 'test': acc_test,
        'test_probs': y_test_probs, 'test_pred': y_test_pred
    }



## Visualization

In [14]:
def plot_history(history, config):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
        ax.plot(history.history[metric],     label='Train', lw=2)
        ax.plot(history.history[f'val_{metric}'], label='Val', lw=2)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(config.MODEL_DIR, 'training_history.png'), dpi=150)
    plt.show()
    best = np.argmax(history.history['val_accuracy'])
    print(f"\nMeilleure val_accuracy : "
          f"{100*history.history['val_accuracy'][best]:.2f}% (epoch {best+1})")


## Main Pipeline

In [ ]:
def main_pipeline(dataset, config, run_name=None, tags=None):
    print("\n" + "="*70)
    print("PIPELINE – ResNet Audio Classifier")
    print("="*70)
    
    run = wandb.init(
        project=    "audio-classifier",
        entity="bau-mat-ucl",
        name=run_name or f"run_{time.strftime('%m%d_%H%M')}",
        tags= tags or [],
        config={
            # Audio
            "sample_rate":   config.SAMPLE_RATE,
            "n_mel":         config.N_MEL,
            "n_fft":         config.N_FFT,
            "hop_length":    config.HOP_LENGTH,
            "duration_ms":   config.DURATION_MS,
            # Architecture
            "dropout":       config.DROPOUT_RATE,
            "l2_reg":        config.L2_REG,
            "label_smooth":  config.LABEL_SMOOTHING,
            "mixup_alpha":   config.MIXUP_ALPHA,
            # Training
            "batch_size":    config.BATCH_SIZE,
            "epochs":        config.EPOCHS,
            "learning_rate": config.LEARNING_RATE,
            "early_stop":    config.EARLY_STOPPING_PATIENCE,
            "reduce_lr_plateau": True,
            "reduce_lr_factor" : 0.5,
            "reduce_lr_patience": 10,
            # Data
            "extra_aug_passes": config.EXTRA_AUG_PASSES
        }
    )
    extractor = FeatureExtractor(config)
    print(f"\nInput shape : {extractor.input_shape}")
    print(f"  N_MEL={config.N_MEL}, n_frames={extractor.n_frames}, "
          f"SR={config.SAMPLE_RATE}Hz, N_FFT={config.N_FFT}")

    X_train, X_val, X_test, y_train, y_val, y_test, classnames = \
        prepare_dataset(dataset, config, extractor, n_augmentations=40)
    print("\n" + "-"*70)
    print("MEMORY CHECK AFTER prepare_dataset")
    print("-"*70)
    print("X_train:", X_train.shape, X_train.dtype, X_train.nbytes / 1024 / 1024, "MB")
    print("y_train:", y_train.shape, y_train.dtype, y_train.nbytes / 1024 / 1024, "MB")
    print("X_val  :", X_val.shape,   X_val.dtype,   X_val.nbytes / 1024 / 1024, "MB")
    print("y_val  :", y_val.shape,   y_val.dtype,   y_val.nbytes / 1024 / 1024, "MB")
    print("X_test :", X_test.shape,  X_test.dtype,  X_test.nbytes / 1024 / 1024, "MB")
    print("y_test :", y_test.shape,  y_test.dtype,  y_test.nbytes / 1024 / 1024, "MB")
    print("-"*70)
    n_classes = len(classnames)
    model = build_resnet_audio(extractor.input_shape, n_classes, config)
    wandb.log({"n_params": model.count_params()})
    history = train_model(model, X_train, y_train, X_val, y_val, config, n_classes)
    plot_history(history, config)

    best_model = keras.models.load_model(
        os.path.join(config.MODEL_DIR, 'best_model.keras'))

    metrics = evaluate_full(
        best_model, X_train, y_train, X_val, y_val, X_test, y_test,
        classnames, config, extractor
    )

    # ── Log des métriques finales ─────────────────────────────
    wandb.log({
        "final/train_acc": metrics['train'],
        "final/val_acc":   metrics['val'],
        "final/test_acc":  metrics['test'],
    })

    artifact = wandb.Artifact("best_model", type="model")
    artifact.add_file(os.path.join(config.MODEL_DIR, 'best_model.keras'))
    run.log_artifact(artifact)

    wandb.finish()


    model_save_path = os.path.join(config.MODEL_DIR, 'best_model.keras')
    best_model.save(model_save_path)
    weights_path = os.path.join(config.MODEL_DIR, 'best_model.weights.h5')
    best_model.save_weights(weights_path)
    meta = {
        'classnames':   classnames,
        'input_shape':  extractor.input_shape,
        'n_mel':        config.N_MEL,
        'n_fft':        config.N_FFT,
        'hop_length':   config.HOP_LENGTH,
        'sample_rate':  config.SAMPLE_RATE,
        'test_accuracy': metrics['test'],
        'keras_version': keras.__version__,
    }
    with open(os.path.join(config.MODEL_DIR, 'model_config.pkl'), 'wb') as f:
        pickle.dump(meta, f)

    print("\n" + "="*70)
    print(f"  Test Accuracy (TTAx{config.TTA_STEPS}) : {100*metrics['test']:.2f}%")
    print("="*70)
    return best_model, metrics, classnames, extractor

In [ ]:
dataset = Dataset()
dataset.remove_class("background")
dataset.remove_class("birds")
dataset.remove_class("handsaw")
dataset.remove_class("helicopter")

model, metrics, classnames, extractor = main_pipeline(dataset, config, run_name=f"resnet_lr{config.LEARNING_RATE}_bs{config.BATCH_SIZE}_{time.strftime('%m%d_%H%M')}", tags=["mateo", "resnet", "more-real-data"])

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/mateobauvir/.netrc.



PIPELINE – ResNet Audio Classifier


wandb: Currently logged in as: guillaume-keus (guillaume-keus-universit-catholique-de-louvain) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Input shape : (64, 87, 1)
  N_MEL=64, n_frames=87, SR=11025Hz, N_FFT=512

PREPA DES DONNEES

[Synthetique - 40 passes d'augmentation]
  Pass 40/40...
  -> Synthetique augmenté : 6560 vecteurs  shape=(6560, 64, 87, 1)

[Acquired training/]
    chainsaw       :   90 fichiers
    fire           :   97 fichiers
    fireworks      :   95 fichiers
    gunshot        :   97 fichiers

  -> Pool acquired : 1516 vecteurs  shape=(1516, 64, 87, 1)

  -> Pool total : 8076 vecteurs  shape=(8076, 64, 87, 1)
  -> Train : 6460  |  Val : 1616

  Augmentation reels (60 passes)...
  -> Train apres aug reels : 97420

[Test reel]
    chainsaw       :   10 fichiers
    fire           :   10 fichiers
    fireworks      :   10 fichiers
    gunshot        :   10 fichiers

[VAL - réel] ../mcu/hands_on_audio_acquisition/audio_files/classname/val/
    chainsaw       :   25 fichiers
    fire           :   25 fichiers
    fireworks      :   25 fichiers
    gunshot        :   25 fichiers

[TRAIN+ - réel] ../mcu/hand

...

 Trainable params: 1,644,516 (6.27 MB)

 Non-trainable params: 4,032 (15.75 KB)


Paramètres : 1,648,548  (~6440 KB FP32)

ENTRAÎNEMENT
  Train : 97420 exemples | 3045 batches/epoch
  Val   : 1616 exemples
  MixUp alpha=0.3 | Label smoothing=0.1
Epoch 1/300


/home/guillaume/LELEC210X-project/.venv/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1425/1425 ━━━━━━━━━━━━━━━━━━━━ 0s 983ms/step - accuracy: 0.4573 - loss: 1.5487 - top_k_categorical_accuracy: 0.6965
Epoch 1: val_accuracy improved from None to 0.73250, saving model to ./data/models/models_resnet/best_model.keras
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 1412s 985ms/step - accuracy: 0.5487 - loss: 1.4000 - top_k_categorical_accuracy: 0.7645 - val_accuracy: 0.7325 - val_loss: 0.9812 - val_top_k_categorical_accuracy: 0.9300
Epoch 2/300
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6994 - loss: 1.1602 - top_k_categorical_accuracy: 0.8645
Epoch 2: val_accuracy improved from 0.73250 to 0.78250, saving model to ./data/models/models_resnet/best_model.keras
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 1477s 1s/step - accuracy: 0.7239 - loss: 1.1148 - top_k_categorical_accuracy: 0.8780 - val_accuracy: 0.7825 - val_loss: 0.9269 - val_top_k_categorical_accuracy: 0.9175
Epoch 3/300
 381/1425 ━━━━━━━━━━━━━━━━━━━━ 16:31 949ms/step - accuracy: 0.7783 - loss: 1.0198 - top_k_categorical_accuracy: 0.9